<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c35-header.png" alt="Nextia Learning · Forecasting and Time Series: Predict What Comes Next" width="100%">

# Solution: Breaks and changes

**[Breaks and changes](https://learning.nextia-ai.com/courses/forecasting/m06/breaks-and-changes/)** · Forecasting and Time Series: Predict What Comes Next · Module 6, lesson 4 of 4 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** see a lasting change in a series (a price rise), check if a simple rule finds it, and test if a shorter training window helps the forecast after it.

| | |
|---|---|
| **Time** | About 30 minutes |
| **Needs** | An internet connection for the setup cells. No account and no key. |
| **You should know** | ETS and its smoothing weight, from [Exponential smoothing](https://learning.nextia-ai.com/courses/forecasting/m04/exponential-smoothing/). WAPE and bias, from [Measure the error that matters](https://learning.nextia-ai.com/courses/forecasting/m03/measure-the-error-that-matters/). |
| **Tested** | Python 3.12 with pandas 2.3.3, statsforecast 2.1.1, mlforecast 1.1.0, hierarchicalforecast 1.5.3, LightGBM 4.7.0 and matplotlib 3.11.2, in Jupyter on a laptop CPU (Apple M4, 16 GB), from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/forecasting/m06/breaks-and-changes/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C35/M06-L04-breaks-and-changes/breaks-and-changes-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs the versions that the course uses: pandas 2.3.3, statsforecast 2.1.1, and matplotlib for the charts. It needs an internet connection and takes about a minute the first time.

You should see one line per package, with the versions above. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again. Restarting is needed when an older version was already loaded.

In [ ]:
%pip install -q --disable-pip-version-check pandas==2.3.3 statsforecast==2.1.1 matplotlib==3.11.2
import warnings; warnings.filterwarnings("ignore")
import pandas; print("pandas", pandas.__version__)
import statsforecast; print("statsforecast", statsforecast.__version__)

## Setup: get the practice data

Run the next cell. It downloads the Larkfield data of this course (about 2 MB) into a folder `larkfield-forecasting/data`, and the course's backtest helper `backtest.py`, and checks each file's checksum. It needs an internet connection and no account, and takes about 10 seconds.

You should see `Ready: 403758 rows of daily sales (200 products, 3 regions), from 2024-04-01 to 2026-03-29.` If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings.
- `wrong checksum`: delete the folder `larkfield-forecasting` (in Colab: the **Files** panel on the left), then run the cell again.

The Larkfield data is synthetic: we made it for this course. It has no real people, products or prices.

In [ ]:
# Setup: download the course data into larkfield-forecasting/data/ and check it.
import gzip, hashlib, os, shutil, subprocess, urllib.request
from pathlib import Path

LABS = "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C35/"
FILES = {
    "calendar.csv": "a5d1d5a3b640a983de72a30fca0e627a4ef23e797b1d80619e9ffcaadf118844",
    "demand.csv.gz": "8b39303985cbdb9bb1f3c705cd927ed6aaeb6f9665bd56ad4c3f8cb2b8578c99",
    "orders.csv": "bd8059fd73fe294daa157f8a6b6ae3f58319fa644eaa7ff0d31122b02cf05e30",
    "price_changes.csv": "889b258be4e933b87065d81d0771d6cd9b630ee90223b0a6ab074129c42153cb",
    "products.csv": "198b0a3de6eea8128af68f3cbc3a5c6acd3a64ac854fff18d37660857be8cfbf",
    "promotions.csv": "ca147959faa4c36c757742de7121976ffe79d59b53049dbbc8337a7547541f4a",
    "roster_hourly.csv": "6c2e0e96881b36ded52ed19458515c71f025288ddaebd589b50c82a8ad7fa69b",
    "tickets_daily.csv": "36f7067db060fdd2bd084c408c0622370a012773849f243020336f1fc678b31d",
    "tickets_hourly.csv.gz": "43f7fbb3e360a8e4a0cb4e101ea6a6242a76f0c37204d76ee9413ea3adbedd92",
    "weather.csv": "43a199a8b235940e291195bfd026901b62a10189b29f3a3ad9527b5382544276",
    "backtest.py": "764ecd38c6e96632be6f0e139e05eadd7b07d14a4ece73d881fc7559c2fa203a",
}
HERE = Path.cwd()
# The notebook works in its own folder, marked with a file, so it never
# writes into your own project if you open the notebook there.
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
    HERE = Path.cwd()
PROJECT = Path("larkfield-forecasting").resolve()
(PROJECT / "data").mkdir(parents=True, exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

def fetch(name, url, path, expected):
    """Use a copy next to the notebook if there is one, or download it."""
    if path.exists() and sha256(path) == expected:
        return
    if (HERE / name).exists():
        shutil.copy(HERE / name, path)
    else:
        download(url, path)
    if sha256(path) != expected:
        raise SystemExit(f"{name} has the wrong checksum. Delete the folder larkfield-forecasting and run again.")

for name, expected in FILES.items():
    if name == "backtest.py":
        fetch(name, LABS + name, PROJECT / name, expected)
    else:
        fetch(name, LABS + "data/" + name, PROJECT / "data" / name, expected)

os.chdir(PROJECT)  # work in the project folder, as in the lesson
with gzip.open("data/demand.csv.gz", "rt") as f:
    header = f.readline()
    rows = [line.split(",", 3)[:3] for line in f]
print(f"Ready: {len(rows)} rows of daily sales ({len({r[2] for r in rows})} products, {len({r[1] for r in rows})} regions), "
      f"from {min(r[0] for r in rows)} to {max(r[0] for r in rows)}.")

## Setup: helpers

The next cell loads pandas as `pd`, numpy as `np` and the course's backtest helper as `bt`, and defines the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.

Run the cell. It prints nothing.

In [ ]:
# Helpers: the backtest helper, and checks that do not show the answer.
import hashlib, json, logging, os, warnings
os.environ["NIXTLA_ID_AS_COL"] = "1"
import numpy as np
import pandas as pd
import backtest as bt
pd.set_option("display.precision", 3)
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 140)
warnings.filterwarnings("ignore")
logging.getLogger("lightning").setLevel(logging.ERROR)

def fingerprint(value):
    def norm(v):
        if hasattr(v, "to_dict") and hasattr(v, "columns"):
            return {"columns": [str(c) for c in v.columns], "rows": [norm(list(r)) for r in v.itertuples(index=False)]}
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 4)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
        return
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

def load_sales(products=None):
    """Daily sales per product, the three regions added up, in the long format of the backtest:
    unique_id (the product), ds (the day), y (units sold), price, on_promotion, stock_out.
    The missing day 2025-10-19 gets the mean of the same weekday one week before and after."""
    d = pd.read_csv("data/demand.csv.gz", parse_dates=["date"])
    if products is not None:
        d = d[d["product_id"].isin(products)]
    g = d.groupby(["product_id", "date"], as_index=False).agg(
        y=("sales", "sum"), price=("price", "mean"), on_promotion=("on_promotion", "max"), stock_out=("stock_out", "max"))
    out = []
    for uid, s in g.rename(columns={"product_id": "unique_id", "date": "ds"}).groupby("unique_id"):
        s = s.set_index("ds").reindex(pd.date_range(s["ds"].min(), s["ds"].max(), freq="D"))
        gap = s["y"].isna()
        before, after = s["y"].shift(7), s["y"].shift(-7)
        s.loc[gap, "y"] = ((before + after) / 2).fillna(before).fillna(after)[gap].round()
        s = s.ffill()
        s["unique_id"] = uid
        out.append(s.rename_axis("ds").reset_index())
    return pd.concat(out, ignore_index=True)[["unique_id", "ds", "y", "price", "on_promotion", "stock_out"]]

# The course backtest (the same in every module): 13 origins, 28 days apart, the last one 2026-03-15.
ORIGINS = bt.origins("2026-03-15", n=13, every="28D")
BACKTEST_PRODUCTS = ['P001', 'P002', 'P005', 'P006', 'P009', 'P013', 'P018', 'P024', 'P025', 'P029', 'P032', 'P039', 'P045', 'P057', 'P068', 'P075', 'P077', 'P078', 'P080', 'P081', 'P087', 'P088', 'P093', 'P095', 'P108', 'P110', 'P121', 'P130', 'P133', 'P136', 'P137', 'P141', 'P145', 'P147', 'P148', 'P156', 'P158', 'P159', 'P166', 'P169', 'P172', 'P174', 'P176', 'P183', 'P184', 'P186', 'P191', 'P192', 'P194', 'P199']
PRICES = pd.read_csv("data/products.csv").set_index("product_id")["base_price"].to_dict()

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. Old days after a change

A product sold 20 units a day for 10 days. Then its price went up, and it sold 12 a day for 5 days. Two forecasts for tomorrow:

- the **mean of all days**: every day counts the same;
- a **smoothed level** with weight 0.3: each new day pulls the level 30% of the way toward it, so old days count less and less.

> **Predict.** Which forecast is closer to 12? Then run the cell.

In [ ]:
days = [20] * 10 + [12] * 5
level = days[0]
for v in days[1:]:
    level = 0.3 * v + 0.7 * level
print("mean of all days:", round(np.mean(days), 1), "| mean since the change:", np.mean(days[10:]),
      "| smoothed level:", round(level, 1), "| weight left on the old level:", round(0.7 ** 5, 3))

> **Check.** You should see `mean of all days: 17.3 | mean since the change: 12.0 | smoothed level: 13.3 | weight left on the old level: 0.168`.

The mean of all days is far too high: it needs a **shorter window**. The smoothed level forgets the old days by itself: after 5 days, they keep 16.8% of the weight.

## 2. The garden hose after its price rise

On 2 June 2025, the garden hose P077 went from 24.99 to 31.99 dollars (`price_changes.csv`). The next cell adds up its sales per week (Monday to Sunday) and compares the 8 weeks before and after the rise with the same weeks one year before.

In [ ]:
hose = load_sales(["P077"])
weeks = hose.set_index("ds")["y"].resample("W-SUN").sum()   # each week ends on a Sunday
rise, last_year = pd.Timestamp("2025-06-02"), pd.Timestamp("2024-06-03")
print(pd.read_csv("data/price_changes.csv").query("product_id == 'P077'").to_string(index=False))
print("2025: 8 weeks before", round(weeks[:rise][-8:].mean(), 1), "| 8 weeks after", round(weeks[rise:][:8].mean(), 1))
print("2024: 8 weeks before", round(weeks[:last_year][-8:].mean(), 1), "| 8 weeks after", round(weeks[last_year:][:8].mean(), 1))

> **Check.** You should see the price change, then `2025: 8 weeks before 126.9 | 8 weeks after 79.8` and `2024: 8 weeks before 106.4 | 8 weeks after 117.8`.

In 2024, June sold more than spring. In 2025, it sold 37% less. The season did not change; the price did.

## 3. Can a simple rule find the change?

A simple rule: an alarm when a week sells less than 75% of the mean of the 8 weeks before it, two weeks in a row.

> **Predict.** Will the rule fire only in June 2025? Then run the cell.

In [ ]:
ratio = weeks / weeks.rolling(8).mean().shift(1)
low = ratio < 0.75
fires = ratio.index[low & low.shift(1, fill_value=False)]
print("alarm in the weeks that end on:", [d.date().isoformat() for d in fires])

> **Check.** You should see 7 weeks: three in late October and November 2024, one in January 2025, two in June 2025 and one at the end of December 2025.

The rule finds the price rise, but it also fires when the season ends. A level rule cannot tell a season from a break. Compare with the same weeks last year, and read the planned changes (prices, channels, suppliers). Finding unusual points and change points in many series is the subject of the anomaly detection course.

## 4. Does a shorter window help ETS?

At three cut-offs after the rise, the next cell fits ETS twice: on all the history, and only on the days since the rise. It forecasts 14 days and scores both.

> **Predict.** Will the window since the rise give a lower WAPE? Then run the cell (a few seconds).

In [ ]:
from statsforecast import StatsForecast
from statsforecast.models import AutoETS

rows = []
for origin in pd.to_datetime(["2025-07-13", "2025-08-10", "2025-09-07"]):
    real = hose[(hose["ds"] > origin) & (hose["ds"] <= origin + pd.Timedelta(days=14))]["y"].to_numpy()
    for window, start in [("all history", hose["ds"].min()), ("since the rise", rise)]:
        train = hose[(hose["ds"] >= start) & (hose["ds"] <= origin)][["unique_id", "ds", "y"]]
        sf = StatsForecast(models=[AutoETS(season_length=7)], freq="D").fit(train)
        f = sf.predict(h=14)["AutoETS"].to_numpy()
        fitted = sf.fitted_[0, 0].model_                      # the model that AutoETS chose
        rows.append({"origin": origin.date(), "window": window, "days": len(train), "model": fitted["method"],
                     "alpha": round(float(fitted["par"][0]), 4),
                     "WAPE %": round(float(100 * np.abs(real - f).sum() / real.sum()), 1),
                     "bias %": round(float(100 * (f - real).sum() / real.sum()), 1)})
ets = pd.DataFrame(rows)
print(ets.to_string(index=False, float_format=lambda v: f"{v:g}"))

> **Check.** You should see six rows. All history: WAPE 23.4, 15.9 and 26.4, with ETS(A,N,A) and an alpha of about 0.15. Since the rise: 24.6, 17.2 and 27.7, with ETS(A,N,N) or ETS(M,N,M) and an alpha of 0.0001.

The shorter window is **worse** at all three cut-offs, by 1.2 to 1.3 points. With only 42 to 98 days, ETS chose no weekly pattern (A,N,N) at two cut-offs and a nearly flat level at all three. On all the history, it kept the weekly pattern, and its level had already moved down.

## 5. Your turn: how much does the old level still count?

> **Your turn.** Take the alpha of the all-history ETS at the first cut-off (`ets.loc[0, "alpha"]`). After the 42 days from the rise to that cut-off, the level before the rise keeps the weight (1 − alpha) to the power 42. Compute it, round it to 4 decimals and put it in `old_weight`. Then run the check cell.

In [ ]:
alpha = ets.loc[0, "alpha"]
old_weight = round(float((1 - alpha) ** 42), 4)
print(alpha, old_weight)

In [ ]:
expect('old_weight', old_weight, 0.0008, tolerance=5e-05)

After 42 days, the old level keeps less than 0.1% of the weight. ETS had already moved on from the old price. A shorter window helps a method that weighs all days the same, such as a mean of all days, not ETS.

## Next

You can now find a lasting change and decide if the training window must change. The [lesson page](https://learning.nextia-ai.com/courses/forecasting/m06/breaks-and-changes/) has the charts, the diagrams and the knowledge checks that count toward your certificate.

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [A forecast is a range](https://learning.nextia-ai.com/courses/forecasting/m07/a-forecast-is-a-range/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/forecasting/m06/breaks-and-changes/)

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The practice data is synthetic and has no real people.